# 01d — Reasoning-effort runs (data generation only)

Run GPT-5 no-desc on `tuning_sample.csv` (the tuning set) at three values of
`reasoning_effort`: `low`, `medium` (default), and `high`.  Three API keys are
used so the three runs go in parallel, avoiding per-key rate limits.

**Outputs (saved to `data/results/llm/`):**
- `01d_predictions.csv` — one row per loan × variant: `actual`, `llm_pred`, `prob_fully_paid`, `reasoning_effort`, `llm_reasoning`
- `01d_metrics.csv` — accuracy / precision / recall / F1 / AUC per variant (default-threshold view)
- Per-call rows are also auto-appended to `llm_calls.csv`.

This notebook **only generates data**.  Threshold tuning, the medium-vs-high
decision, and the held-out validation all happen in `06_threshold_tune_and_test.ipynb`.

In [ ]:
# llm_utils.py and llm_pricing.py live one directory up — make them importable.
import sys; sys.path.insert(0, "..")

import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    compare_results, evaluate_predictions, RESULTS_DIR,
)

In [ ]:
# Load OpenAI API keys from .env.
# We have two keys; assign them round-robin across the three effort levels so
# the parallel runs don't hammer a single rate-limit bucket.
load_dotenv("../.env", override=False)

_key1 = os.environ.get("OPENAI_API_KEY")
_key2 = os.environ.get("OPENAI_API_KEY_2") or _key1   # fallback to key1 if key2 missing

API_KEYS = {
    "low":    _key1,
    "medium": _key2,
    "high":   _key1,   # low and high share key1 — different efforts rarely collide
}
missing = [k for k, v in API_KEYS.items() if not v]
assert not missing, f"Missing API keys for: {missing}. Add them to ../.env."
print("API keys loaded (key1 used for low+high, key2 for medium).")

In [ ]:
# Tuning sample = the 100-loan LLM eval sample (built by 02_Preprocessing).
# Threshold tuning will use these labels in the next notebook.
tuning_sample = load_llm_sample()
y_true = tuning_sample['loan_status'].values
print(f"Tuning sample: {len(tuning_sample)} loans")
print(f"  Charged Off: {(y_true == 0).sum()}")
print(f"  Fully Paid:  {(y_true == 1).sum()}")

# XGBoost predictions on the same sample for context (printed only).
xgb_probs, xgb_preds = run_ml_on_sample(tuning_sample)
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(),
                                   label="XGBoost (tuning sample)",
                                   probabilities=xgb_probs.tolist())

In [ ]:
# Run three GPT-5.4 variants concurrently, each with its own API key.
MODEL = "gpt-5.4"
EFFORTS = ["low", "medium", "high"]

def run_one(effort):
    return run_llm_experiment(
        tuning_sample,
        api_provider="openai",
        model_name=MODEL,
        api_key=API_KEYS[effort],
        label=f"GPT-5.4 reasoning={effort}",
        include_desc=False,
        with_logprobs=True,
        reasoning_effort=effort,
    )

results = {}
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(run_one, e): e for e in EFFORTS}
    for fut in futures:
        e = futures[fut]
        results[e] = fut.result()

print("\nAll three runs complete.")

In [ ]:
# Build a single consolidated predictions CSV (one row per loan × variant).
rows = []
for effort, res in results.items():
    for i in range(len(tuning_sample)):
        rows.append({
            "row_index":         i,
            "reasoning_effort":  effort,
            "actual":            int(y_true[i]),
            "llm_pred":          res['predictions'][i],
            "prob_fully_paid":   res['probabilities'][i],
            "llm_reasoning":     res['reasonings'][i],
            "xgb_pred":          int(xgb_preds[i]),
            "xgb_prob":          float(xgb_probs[i]),
        })

predictions_df = pd.DataFrame(rows)
out_path = f"{RESULTS_DIR}/01d_predictions.csv"
predictions_df.to_csv(out_path, index=False)
print(f"Saved {len(predictions_df)} rows to {out_path}")

In [ ]:
# Summary metrics per variant at the LLM's default (hard 0/1) prediction.
# These are the BEFORE-tuning numbers; tuned-threshold metrics are in 06.
metrics_rows = []
for effort, res in results.items():
    m = res['metrics'].copy()
    m['reasoning_effort'] = effort
    m['variant'] = f"GPT-5 reasoning={effort}"
    metrics_rows.append(m)

metrics_rows.append({**xgb_metrics, 'reasoning_effort': '-', 'variant': 'XGBoost (tuning)'})

metrics_df = pd.DataFrame(metrics_rows)
cols = ['variant', 'reasoning_effort', 'accuracy', 'auc',
        'precision_charged_off', 'recall_charged_off', 'f1_charged_off']
metrics_df = metrics_df[[c for c in cols if c in metrics_df.columns]]

out_path = f"{RESULTS_DIR}/01d_metrics.csv"
metrics_df.to_csv(out_path, index=False)
print(metrics_df.to_string(index=False))
print(f"\nSaved metrics to {out_path}")